In [29]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler

import numpy as np
import matplotlib.pyplot as plt

import scienceplots

plt.style.use(["science", "notebook", "grid"])

import numpy as np
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

In [30]:
import pickle

In [31]:
from models.models import LeNet5
from visualization.filters import display_filters

In [57]:
from datasets.data_modules import CustomImageDataModule

In [32]:
import pytorch_lightning as pl

In [33]:
from train.train import get_features, train_model

In [34]:
my_local_data = "/mnt/g/My Drive/types/"

In [35]:
ToTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)
aux_data = datasets.ImageFolder(root=my_local_data, transform=ToTensorAndNormalize)

In [36]:
# class_indices = aux_data.class_to_idx
# # Original names and their order which you know
# names_dict = {
#     "typeA": "Diameter\nFluctuations",
#     "typeB": "Node Cut",
#     "typeC": "Particle Hit",
#     "goodIngots": "No Structure\nLoss",
# }

# # Get class indices from the ImageFolder
# class_indices = aux_data.class_to_idx

# # Order your names list according to the class indices
# ordered_names = [
#     names_dict[class_name]
#     for class_name, index in sorted(class_indices.items(), key=lambda item: item[1])
# ]

In [37]:
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# model = LeNet5()
# # model = model.to(device=device)
# batch_size = 16

In [38]:
# device

In [39]:
# # Determine the layer from which to extract features --> we want to extract at the pooling layers
# layers = ["convStack.1", "convStack.3"]
# node_names = get_graph_node_names(model)
# loader = DataLoader(
#     dataset=aux_data, batch_size=batch_size, shuffle=False, num_workers=8
# )
# features, labels = get_features(
#     model=model, layers=layers, data_loader=loader, device=device
# )
# pooling_layers = ["Pooling layer 1", "Pooling layer 2"]
# display_filters(
#     features=features,
#     img_num=156,
#     layers=layers,
#     cmap="viridis",
#     data=aux_data,
#     name_layers=pooling_layers,
# )

In [40]:
# Try to load cached targets first
try:
    with open(f"logdir/cached_targets.pkl", "rb") as f:
        targets = pickle.load(f)
except FileNotFoundError:
    targets = [t for _, t in aux_data]
    # Cache the targets for next time
    with open(f"logdir/cached_targets.pkl", "wb") as f:
        pickle.dump(targets, f)

In [41]:
from lightning_modules.lightning_modules import LitLeNet5

In [42]:
model2 = LitLeNet5(num_classes=4,
                   learning_rate=0.001)

In [43]:
from torchvision import datasets

dataset = datasets.ImageFolder(my_local_data)
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class to index mapping: {dataset.class_to_idx}")

Number of classes: 4
Class to index mapping: {'goodIngots': 0, 'typeA': 1, 'typeB': 2, 'typeC': 3}


In [45]:
from sklearn.model_selection import StratifiedKFold

In [46]:
kfold = StratifiedKFold(
    n_splits=4,
    shuffle=True,
)

In [47]:
validation_metrics = []

In [48]:
trainer_config = {
    'patience': 3,
    'accelerator': 'gpu',
    'devices': -1,
    'max_epochs': 10,
    'precision': 32,
    'n_steps':5
}

In [49]:
train_size = int(len(aux_data)*0.8)
val_size = len(aux_data) - train_size

In [50]:
from sklearn.model_selection import train_test_split
import torch

# Assuming aux_data is a dataset object and targets are the labels
train_idx, val_idx, _, _ = train_test_split(
    range(len(aux_data)), targets, test_size=0.2, random_state=42
)

train_data = torch.utils.data.Subset(aux_data, train_idx)
val_data = torch.utils.data.Subset(aux_data, val_idx)

In [51]:
len(val_idx)

51

In [52]:
data_module = CustomImageDataModule(
    train_dataset=train_data,
    val_dataset=val_data,
    batch_size=16,
    num_workers=4,
)

In [53]:
data_module

In [54]:
%%capture
# Assuming model2 is initialized, and trainer_config is defined
val_metrics = train_model(
    model=model2,
    trainer_config=trainer_config,
    save_dir='logdir/',
    data_module=data_module,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params
------------------------------------------------
0 | accuracy | MulticlassAccuracy | 0     
1 | f1_score | MulticlassF1Score  | 0     
2 | loss_fn  | CrossEntropyLoss   | 0     
3 | model    | LeNet5             | 29.4 M
------------------------------------------------
29.4 M    Trainable params
0         Non-trainable params
29.4 M    Total params
117.782   Total estimated model params size (MB)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [55]:
print(val_metrics)

{'val_loss': tensor(1.3760), 'val_accuracy': tensor(0.3137), 'val_f1_score': tensor(0.3137)}


In [27]:
for fold, (train_idx, val_idx) in enumerate(kfold.split(aux_data, targets)):
    
    train_data = torch.utils.data.Subset(aux_data, train_idx)
    val_data = torch.utils.data.Subset(aux_data, val_idx)

    data_module = CustomImageDataModule(
        train_dataset=train_data,
        val_dataset=val_data,
        batch_size=4,
        num_workers=12,
    )

    val_metrics_fold = train_model(model=model2,
                                   trainer_config=trainer_config,
                                   save_dir='logdir/',
                                   data_module=data_module,)

    validation_metrics.append(val_metrics_fold)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


/home/alfredosg/.envs/sl-ml/lib/python3.8/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:639: Checkpoint directory logdir/ exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params
------------------------------------------------
0 | accuracy | MulticlassAccuracy | 0     
1 | f1_score | MulticlassF1Score  | 0     
2 | loss_fn  | CrossEntropyLoss   | 0     
3 | model    | LeNet5             | 29.4 M
------------------------------------------------
29.4 M    Trainable params
0         Non-trainable params
29.4 M    Total params
117.782   Total estimated model params size (MB)


Epoch 3: 100%|██████████| 48/48 [00:16<00:00,  2.86it/s, v_num=3, val_loss=1.390, val_accuracy=0.234, val_f1_score=0.234]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation DataLoader 0: 100%|██████████| 16/16 [00:00<00:00, 18.08it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      val_accuracy               0.234375
      val_f1_score               0.234375
        val_loss            1.3883614540100098
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Fold None Validation Metrics: {'val_loss': tensor(1.3884), 'val_accuracy': tensor(0.2344), 'val_f1_score': tensor(0.2344)}


...

Validation DataLoader 0: 100%|██████████| 16/16 [00:00<00:00, 16.72it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      val_accuracy               0.234375
      val_f1_score               0.234375
        val_loss             1.386082410812378
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params
------------------------------------------------
0 | accuracy | MulticlassAccuracy | 0     
1 | f1_score | MulticlassF1Score  | 0     
2 | loss_fn  | CrossEntropyLoss   | 0     
3 | model    | LeNet5             | 29.4 M
------------------------------------------------
29.4 M    Trainable params
0         Non-trainable params
29.4 M    Total params
117.782   Total estimated model params size (MB)


Fold None Validation Metrics: {'val_loss': tensor(1.3861), 'val_accuracy': tensor(0.2344), 'val_f1_score': tensor(0.2344)}
Epoch 3: 100%|██████████| 48/48 [00:16<00:00,  2.89it/s, v_num=6, val_loss=1.390, val_accuracy=0.270, val_f1_score=0.270]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation DataLoader 0: 100%|██████████| 16/16 [00:01<00:00, 13.54it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      val_accuracy          0.2698412835597992
      val_f1_score          0.2698412835597992
        val_loss            1.3857234716415405
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Fold None Validation Metrics: {'val_loss': tensor(1.3857), 'val_accuracy': tensor(0.2698), 'val_f1_score': tensor(0.2698)}
